## KAT-ICL (Milestone) Report — ACT Transfer Cube

Executable report for the milestone pipeline:

- Demos: `record_sim_episodes.py` (downsample 10 and 5)
- Schema: JSONL episode structure reports
- Obs tokens: top-view keypoints (ViT-DINO via `timm`)
- Action tokens: waypoint-quantized 14D joint+gripper actions
- LLM: Ollama `127.0.0.1:11434` model `gemma4:26b`
- Replay: decode → replay → reward==4 success


In [1]:
import sys, subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()

def run(cmd: str, timeout_s: int = 3600):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True, cwd=str(REPO_ROOT), timeout=timeout_s)

print('python:', sys.executable)
print('repo:', REPO_ROOT)


python: /opt/homebrew/opt/python@3.9/bin/python3.9
repo: /Users/sushantsubedi/Documents/act


### 1) Collect demos (downsample 10 and 5)

Creates HDF5 episodes and renders `episode_0_video.mp4` + `episode_0_qpos.png`.


In [2]:
run(r'''
source .venv/bin/activate && \
# Record ONE base dataset at downsample=10 WITH DEPTH (25 episodes total: 20 demos + 5 tests) \
mkdir -p data/transfer_cube/ds10/base_25_env0_depth && \
python3 record_sim_episodes.py --task_name sim_transfer_cube_scripted --dataset_dir data/transfer_cube/ds10/base_25_env0_depth --num_episodes 25 --downsample_rate 10 && \
python3 visualize_episodes.py --dataset_dir data/transfer_cube/ds10/base_25_env0_depth --episode_idx 0 && \
# Create subsets without re-recording (hardlinks/copies) \
python3 scripts/make_ds10_subsets.py --base_dir data/transfer_cube/ds10/base_25_env0_depth --out_root data/transfer_cube/ds10/subsets_depth
''')


$ 
source .venv/bin/activate && \
# Record ONE base dataset at downsample=10 WITH DEPTH (25 episodes total: 20 demos + 5 tests) \
mkdir -p data/transfer_cube/ds10/base_25_env0_depth && \
python3 record_sim_episodes.py --task_name sim_transfer_cube_scripted --dataset_dir data/transfer_cube/ds10/base_25_env0_depth --num_episodes 25 --downsample_rate 10 && \
python3 visualize_episodes.py --dataset_dir data/transfer_cube/ds10/base_25_env0_depth --episode_idx 0 && \
# Create subsets without re-recording (hardlinks/copies) \
python3 scripts/make_ds10_subsets.py --base_dir data/transfer_cube/ds10/base_25_env0_depth --out_root data/transfer_cube/ds10/subsets_depth

episode_idx=0
Rollout out EE space scripted policy
episode_idx=0 Successful, episode_return=628
Replaying joint commands
episode_idx=0 Successful, episode_return=639
Saving: 0.0 secs

episode_idx=1
Rollout out EE space scripted policy
episode_idx=1 Successful, episode_return=627
Replaying joint commands
episode_idx=1 Successful, epi

### 2) Schema report (monitor episode structure)


In [3]:
run(r'''
source .venv/bin/activate && \
python3 scripts/schema_report.py --dataset_dir data/transfer_cube/ds10/demos_2_env0 --out artifacts/schema_report_ds10_env0.jsonl && \
python3 scripts/schema_report.py --dataset_dir data/transfer_cube/ds5/demos_2 --out artifacts/schema_report_ds5.jsonl
''')


$ 
source .venv/bin/activate && \
python3 scripts/schema_report.py --dataset_dir data/transfer_cube/ds10/demos_2_env0 --out artifacts/schema_report_ds10_env0.jsonl && \
python3 scripts/schema_report.py --dataset_dir data/transfer_cube/ds5/demos_2 --out artifacts/schema_report_ds5.jsonl



Traceback (most recent call last):
  File "/Users/sushantsubedi/Documents/act/scripts/schema_report.py", line 20, in <module>
    main()
  File "/Users/sushantsubedi/Documents/act/scripts/schema_report.py", line 16, in main
    write_schema_report(args.dataset_dir, args.out)
  File "/Users/sushantsubedi/Documents/act/act_kat/episode_schema.py", line 222, in write_schema_report
    episode_paths = _iter_episode_paths(dataset_dir)
  File "/Users/sushantsubedi/Documents/act/act_kat/episode_schema.py", line 214, in _iter_episode_paths
    for name in sorted(os.listdir(dataset_dir)):
FileNotFoundError: [Errno 2] No such file or directory: 'data/transfer_cube/ds10/demos_2_env0'


CalledProcessError: Command '
source .venv/bin/activate && \
python3 scripts/schema_report.py --dataset_dir data/transfer_cube/ds10/demos_2_env0 --out artifacts/schema_report_ds10_env0.jsonl && \
python3 scripts/schema_report.py --dataset_dir data/transfer_cube/ds5/demos_2 --out artifacts/schema_report_ds5.jsonl
' returned non-zero exit status 1.

### 3) Tokenize observations (top-view keypoints)

Runs keypoint tokenization for K=10 and K=20 on one episode.


In [ ]:
run(r'''
source .venv/bin/activate && \
python3 scripts/tokenize_obs.py --episode_hdf5 data/transfer_cube/ds10/demos_2_env0/episode_0.hdf5 --k 10 --obs_bins 64 --device cpu && \
python3 scripts/tokenize_obs.py --episode_hdf5 data/transfer_cube/ds10/demos_2_env0/episode_0.hdf5 --k 20 --obs_bins 64 --device cpu
''')


In [7]:
run(r'''
source .venv/bin/activate && \
# Visualize extracted top-view keypoints (overlay + printed tokens)
python3 scripts/visualize_keypoints.py \
  --episode_hdf5 data/transfer_cube/ds10/base_25_env0_depth/episode_0.hdf5 \
  --t 0 --k 10 --obs_bins 64 --device cpu \
  --out_png artifacts/keypoints_overlay_ep0_t0_k10.png
''')


$ 
source .venv/bin/activate && \
# Visualize extracted top-view keypoints (overlay + printed tokens)
python3 scripts/visualize_keypoints.py \
  --episode_hdf5 data/transfer_cube/ds10/base_25_env0_depth/episode_0.hdf5 \
  --t 0 --k 10 --obs_bins 64 --device cpu \
  --out_png artifacts/keypoints_overlay_ep0_t0_k10.png

Saved overlay: artifacts/keypoints_overlay_ep0_t0_k10.png
Tokens:
OBS_START
KP 0 x=56 y=34 d=63
KP 1 x=43 y=47 d=25
KP 2 x=47 y=43 d=16
KP 3 x=11 y=25 d=25
KP 4 x=20 y=38 d=25
KP 5 x=38 y=25 d=25
KP 6 x=34 y=43 d=25
KP 7 x=16 y=34 d=15
KP 8 x=11 y=43 d=25
KP 9 x=16 y=29 d=25
OBS_END



### 4) Tokenize actions (waypoints)


In [8]:
run(r'''
source .venv/bin/activate && \
python3 scripts/tokenize_action.py --episode_hdf5 data/transfer_cube/ds10/demos_2_env0/episode_0.hdf5 --M 20
''')


$ 
source .venv/bin/activate && \
python3 scripts/tokenize_action.py --episode_hdf5 data/transfer_cube/ds10/demos_2_env0/episode_0.hdf5 --M 20



Traceback (most recent call last):
  File "/Users/sushantsubedi/Documents/act/scripts/tokenize_action.py", line 31, in <module>
    main()
  File "/Users/sushantsubedi/Documents/act/scripts/tokenize_action.py", line 20, in main
    with h5py.File(args.episode_hdf5, "r") as root:
  File "/Users/sushantsubedi/Documents/act/.venv/lib/python3.9/site-packages/h5py/_hl/files.py", line 567, in __init__
    fid = make_fid(name, mode, userblock_size, fapl, fcpl, swmr=swmr)
  File "/Users/sushantsubedi/Documents/act/.venv/lib/python3.9/site-packages/h5py/_hl/files.py", line 231, in make_fid
    fid = h5f.open(name, flags, fapl=fapl)
  File "h5py/_objects.pyx", line 54, in h5py._objects.with_phil.wrapper
  File "h5py/_objects.pyx", line 55, in h5py._objects.with_phil.wrapper
  File "h5py/h5f.pyx", line 106, in h5py.h5f.open
FileNotFoundError: [Errno 2] Unable to open file (unable to open file: name = 'data/transfer_cube/ds10/demos_2_env0/episode_0.hdf5', errno = 2, error message = 'No such file o

CalledProcessError: Command '
source .venv/bin/activate && \
python3 scripts/tokenize_action.py --episode_hdf5 data/transfer_cube/ds10/demos_2_env0/episode_0.hdf5 --M 20
' returned non-zero exit status 1.

### 5) Call Ollama (REAL ICL, prompt+response logged)

This runs **real ICL**: the LLM must generate the action-token codes.

Implementation detail: we use `--mode icl_json` which calls Ollama `/api/generate` with a **JSON schema constraint** so the output is parseable.

Outputs are logged under `artifacts/` and the generated JSON is also written to `artifacts/icl_json_outputs/<run_id>.json`.


In [ ]:
run(r'''
source .venv/bin/activate && \
python3 scripts/run_icl.py --dataset_dir data/transfer_cube/ds10/demos_2_env0 --demo_ids 0 --query_id 1 --K 10 --M 20 --model gemma4:26b --ollama_url http://127.0.0.1:11434 --device cpu --run_id icl_json_notebook_ds10_ep1 --mode icl_json
''')


### 6) Decode + replay (success check)

Writes a quantizer JSON then decodes + replays and saves a video.


In [ ]:
run(r'''
source .venv/bin/activate && \
python3 - <<'PY'
import json, os
import numpy as np
import h5py
os.makedirs('artifacts/quantizers', exist_ok=True)
ds='data/transfer_cube/ds10/demos_2_env0'
acts=[]
for i in [0,1]:
  with h5py.File(f'{ds}/episode_{i}.hdf5','r') as r:
    acts.append(r['/action'][()].astype(np.float32))
all_act=np.concatenate(acts,axis=0)
path='artifacts/quantizers/notebook_ds10_env0_bins64.json'
json.dump({'mins':all_act.min(axis=0).tolist(),'maxs':all_act.max(axis=0).tolist()}, open(path,'w'), indent=2)
print(path)
PY
''')

run(r'''
source .venv/bin/activate && \
python3 scripts/replay_from_icl_json.py \
  --icl_json artifacts/icl_json_outputs/icl_json_notebook_ds10_ep1.json \
  --quantizer_json artifacts/quantizers/notebook_ds10_env0_bins64.json \
  --bins 64 --episode_len 400 \
  --video_path artifacts/replays/icl_json_notebook_ds10_ep1.mp4 \
  --box_pose_episode_hdf5 data/transfer_cube/ds10/demos_2_env0/episode_1.hdf5
''')


### 7) Evaluation (REAL ICL success rate)

This computes the **real ICL** success rate by:
- building few-shot prompts (N demos)
- forcing LLM output into a parseable JSON schema
- decoding + replaying each generated action sequence

Outputs:
- per-episode rollouts: `artifacts/replays/eval_icl_json_ep*.mp4`
- per-episode CSV: `artifacts/results_icl_json.csv`
- printed summary JSON with `success_rate`



In [ ]:
run(r'''
source .venv/bin/activate && \
# Evaluate REAL ICL (DEPTH-enabled integer keypoint tokens) using 5/10/20 demo subsets against the held-out test subset.
# Hold/repeat rollout: the sim runs `episode_len` steps by holding each of the M predicted waypoints (no linear interpolation).
python3 scripts/eval_icl_json_folders.py \
  --demos_dir data/transfer_cube/ds10/subsets_depth/demos_5 \
  --tests_dir data/transfer_cube/ds10/subsets_depth/test_5 \
  --K 10 --M 20 --bins 64 --obs_bins 64 --episode_len 400 --out_csv artifacts/results_icl_json_depth_demos5.csv ''')

$ 
source .venv/bin/activate && \
# Evaluate REAL ICL (DEPTH-enabled integer keypoint tokens) using 5/10/20 demo subsets against the held-out test subset.
# Hold/repeat rollout: the sim runs `episode_len` steps by holding each of the M predicted waypoints (no linear interpolation).
python3 scripts/eval_icl_json_folders.py \
  --demos_dir data/transfer_cube/ds10/subsets_depth/demos_10 \
  --tests_dir data/transfer_cube/ds10/subsets_depth/test_5 \
  --K 10 --M 20 --bins 1024 --obs_bins 512 --episode_len 400 --out_csv artifacts/results_icl_json_depth_demos5.csv 


/Users/sushantsubedi/Documents/act/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
Traceback (most recent call last):
  File "/Users/sushantsubedi/Documents/act/scripts/eval_icl_json_folders.py", line 234, in <module>
    main()
  File "/Users/sushantsubedi/Documents/act/scripts/eval_icl_json_folders.py", line 149, in main
    resp = ollama_generate(
  File "/Users/sushantsubedi/Documents/act/act_kat/ollama_client.py", line 93, in ollama_generate
    r = requests.post(f"{url}/api/generate", json=payload, timeout=timeout_s)
  File "/Users/sushantsubedi/Documents/act/.venv/lib/python3.9/site-packages/requests/api.py", line 115, in post
    return request("post", url, data=data, json=json, **kwargs)
  File "/Users/sushantsubedi/Documents/act/.venv/lib/python3.9/site-packages/requests/api.py"

KeyboardInterrupt: 

In [ ]:
run(r'''
source .venv/bin/activate && \
# Evaluate REAL ICL (DEPTH-enabled integer keypoint tokens) using 5/10/20 demo subsets against the held-out test subset.
# Hold/repeat rollout: the sim runs `episode_len` steps by holding each of the M predicted waypoints (no linear interpolation).
python3 scripts/eval_icl_json_folders.py \
  --demos_dir data/transfer_cube/ds10/subsets_depth/demos_5 \
  --tests_dir data/transfer_cube/ds10/subsets_depth/test_5 \
  --K 10 --M 20 --bins 64 --obs_bins 64 --episode_len 400 --out_csv artifacts/results_icl_json_depth_demos5.csv && \
python3 scripts/eval_icl_json_folders.py \
  --demos_dir data/transfer_cube/ds10/subsets_depth/demos_10 \
  --tests_dir data/transfer_cube/ds10/subsets_depth/test_5 \
  --K 10 --M 20 --bins 64 --obs_bins 64 --episode_len 400 --out_csv artifacts/results_icl_json_depth_demos10.csv && \
python3 scripts/eval_icl_json_folders.py \
  --demos_dir data/transfer_cube/ds10/subsets_depth/demos_20 \
  --tests_dir data/transfer_cube/ds10/subsets_depth/test_5 \
  --K 10 --M 20 --bins 64 --obs_bins 64 --episode_len 400 --out_csv artifacts/results_icl_json_depth_demos20.csv
''')
